In [ ]:
import os
import warnings
import gc
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf

tf.get_logger().setLevel('ERROR')
tf.get_logger().setLevel("ERROR")
warnings.simplefilter("ignore")

from optuna.storages import RetryFailedTrialCallback
import json
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, multiply, Conv2D, Dense, BatchNormalization, LayerNormalization, Input, MaxPool2D, Dropout, Flatten
import numpy as np
import keras
from phd.Augmentations.common_code.augmentation_functions import *
from phd.Augmentations.common_code.get_dataset import *
from phd.Augmentations.common_code.models import *
from phd.Augmentations.common_code.training_loop import *
from phd.Augmentations.common_code.utils import *

@keras.saving.register_keras_serializable()
class FocalConv(keras.layers.Layer):
    def __init__(self, filters, kernel_size, factor, activation):
        super().__init__()
        self.filters = filters
        self.kernel = kernel_size
        self.activation = activation
        self.factor = factor
        self.conv1 = Conv2D(filters=self.filters, kernel_size=self.kernel, activation=self.activation, padding="same")

    def build(self, input_shape):
        self.gauss = self.make_gaussian(size=input_shape[1], depth=self.filters, factor=self.factor)

    def call(self, x, training=False):
      x = self.conv1(x, training=training)
      batch_size = tf.shape(x)[0]
      xx = tf.repeat(self.gauss, repeats=batch_size, axis=0)
      x = multiply([x, xx])

      return x
    
    def get_config(self):
       return {
          "filters": self.filters,
          "kernel_size": self.kernel,
          "activation": self.activation,
          "factor": self.factor
       }

    def make_gaussian(self, size: int, depth: int, factor: float):
        """ Make a square gaussian kernel.

        size is the length of a side of the square
        fwhm is full-width-half-maximum, which
        can be thought of as an effective radius.
        """
        fwhm = size // 2
        x = np.arange(0, size, 1, float)
        y = x[:,np.newaxis]

        x0 = y0 = size // 2

        gauss = np.exp(-1 * factor * np.log(2) * ((x-x0) ** 2 + (y-y0) ** 2) / fwhm **2)

        return tf.expand_dims(np.stack([gauss for _ in range(depth)], -1), axis=0)


def cavanagh_no_focal(training_config, experiment_name):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))
    
  model.add(Conv2D(filters=32, kernel_size=7, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=64, kernel_size=5, activation='relu', padding='same'))
  model.add(Conv2D(filters=64, kernel_size=5, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=128, kernel_size=3, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Flatten())
  model.add(Dropout(0.5))

  model.add(Dense(256, activation='relu'))
  model.add(Dense(256, activation='relu'))
  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

  return model

def cavanagh_focal_10(training_config, experiment_name):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(FocalConv(filters=32, kernel_size= 7, factor = 10, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=64, kernel_size= 5, factor = 10, activation='relu'))
  model.add(FocalConv(filters=64, kernel_size= 5, factor = 10, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=128, kernel_size=3, factor = 10, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Flatten())
  model.add(Dropout(0.5))

  model.add(Dense(256, activation='relu'))
  model.add(Dense(256, activation='relu'))
  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

  return model

def top_10_loss_4class_original(training_config, experiment_name):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(FocalConv(filters=64, kernel_size= 7, factor = 8, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=64, kernel_size= 5, factor = 5, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=32, kernel_size=7, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=32, kernel_size=5, activation='relu', padding='same')) # wave replaced with regular
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))
    
  model.add(Flatten())
  model.add(Dropout(0.5))

  model.add(Dense(32, activation='relu'))
  model.add(Dense(160, activation='relu'))
  model.add(Dense(160, activation='relu'))
  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

  return model

def top_10_loss_4class_all_focal(training_config, experiment_name):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(FocalConv(filters=64, kernel_size= 7, factor = 10, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=64, kernel_size= 5, factor = 10, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=32, kernel_size=7, factor = 10, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=32, kernel_size=5, factor = 10, activation='relu')) # wave replaced with regular
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))
    
  model.add(Flatten())
  model.add(Dropout(0.5))

  model.add(Dense(32, activation='relu'))
  model.add(Dense(160, activation='relu'))
  model.add(Dense(160, activation='relu'))
  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

  return model
    
def top_10_loss_4class_no_focal(training_config, experiment_name):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(Conv2D(filters=64, kernel_size= 7, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=64, kernel_size= 5, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=32, kernel_size=7, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))
    
  model.add(Conv2D(filters=32, kernel_size=5, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))
  
  model.add(Flatten())
  model.add(Dropout(0.5))

  model.add(Dense(32, activation='relu'))
  model.add(Dense(160, activation='relu'))
  model.add(Dense(160, activation='relu'))
  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

  return model

def best_loss_11class_original(training_config, experiment_name):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(FocalConv(filters=48, kernel_size= 3, factor = 10, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=32, kernel_size= 7, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=48, kernel_size=7, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=32, kernel_size=3, activation='relu', padding='same')) # wave replaced with regular
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))
    
  model.add(Flatten())
  model.add(Dropout(0.5))

  model.add(Dense(224, activation='relu'))
  model.add(Dense(192, activation='relu'))
  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

  return model

def best_loss_11class_all_focal(training_config, experiment_name):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(FocalConv(filters=48, kernel_size= 3, factor = 10, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=32, kernel_size= 7, factor = 10, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=48, kernel_size=7, factor = 10, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=32, kernel_size=3, factor = 10, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))
    
  model.add(Flatten())
  model.add(Dropout(0.5))

  model.add(Dense(224, activation='relu'))
  model.add(Dense(192, activation='relu'))
  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

  return model

def best_loss_11class_no_focal(training_config, experiment_name):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(Conv2D(filters=48, kernel_size= 3, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=32, kernel_size= 7, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=48, kernel_size=7, activation='relu', padding='same'))
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=32, kernel_size=3, activation='relu', padding='same')) # wave replaced with regular
  model.add(BatchNormalization())
  model.add(MaxPool2D(2))
    
  model.add(Flatten())
  model.add(Dropout(0.5))

  model.add(Dense(224, activation='relu'))
  model.add(Dense(192, activation='relu'))
  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

  return model

def cheng_no_focal(training_config, experiment_name):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(Conv2D(filters=32, kernel_size= 3, activation='relu', padding='same'))
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=64, kernel_size= 3, activation='relu', padding='same'))
  model.add(MaxPool2D(2))

  model.add(Conv2D(filters=128, kernel_size=2, activation='relu', padding='same'))
  model.add(MaxPool2D(2))
    
  model.add(Flatten())
  model.add(Dropout(0.5))

  model.add(Dense(1024, activation='relu'))
  model.add(Dense(1024, activation='relu'))

  model.add(Dropout(0.5))

  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

  return model

def cheng_focal_10(training_config, experiment_name):
  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  model.add(FocalConv(filters=32, kernel_size= 3, factor=10, activation='relu'))
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=64, kernel_size= 3, factor=10, activation='relu'))
  model.add(MaxPool2D(2))

  model.add(FocalConv(filters=128, kernel_size=2, factor=10, activation='relu'))
  model.add(MaxPool2D(2))
    
  model.add(Flatten())
  model.add(Dropout(0.5))

  model.add(Dense(1024, activation='relu'))
  model.add(Dense(1024, activation='relu'))

  model.add(Dropout(0.5))

  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

  return model

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import matplotlib.pyplot as plt
import os
from tqdm.notebook import tqdm
import pandas as pd
from pathlib import Path
import numpy as np
import tensorflow as tf
from glob import glob
import sys


from phd.Augmentations.common_code.training_loop import get_dataset_with_objids, get_objids_only
from phd.Augmentations.common_code.utils import reset_tpu
from phd.Augmentations.common_code.models import Cavanagh, create_ResNet50



model_funcs = { 
    'cavanagh_no_focal':cavanagh_no_focal,
    'cavanagh_focal_10':cavanagh_focal_10,
    'top_10_loss_4class_original':top_10_loss_4class_original,
    'top_10_loss_4class_all_focal':top_10_loss_4class_all_focal,
    'top_10_loss_4class_no_focal':top_10_loss_4class_no_focal,
    'best_loss_11class_original':best_loss_11class_original,
    'best_loss_11class_all_focal':best_loss_11class_all_focal,
    'best_loss_11class_no_focal':best_loss_11class_no_focal,
    'cheng_no_focal':cheng_no_focal,
    'cheng_focal_10':cheng_focal_10,
}

training_config = {
    "TPU": False,
    "IMAGE_SIZE": 128,
    "ENABLE_DETERMINISM": False,
    "NUMBER_OF_CLASSES": 2
}

tf.config.run_functions_eagerly(True)

dataset_names = [
    ("paper3_focal_last_Bin_SDSS", "Bin_SDSS_z_0.0-0.25"),
    ("paper3_focal_last_Bin_DES", "Bin_DES_z_0.0-0.15"),
    ("paper3_focal_last_Bin_H_1", "Bin_Hubble_z_0.0-0.3"),
    ("paper3_focal_last_Bin_H_2", "Bin_Hubble_z_0.3-1.0"),
    ("paper3_focal_last_Bin_H_3", "Bin_Hubble_z_1.0_4.0"),
]

for (experiment_name, dataset_name) in dataset_names:
    model_paths = glob(f"models/128x128/experiments/{experiment_name}/*")
    print(model_paths)
    for model_path in model_paths:
        print(model_path)
        model_path = model_path.replace("\\", "/")
        parts = model_path.split("/")
        model_name = parts[4]
        accuracies = []
        precisions = []
        recalls = []
        f1s = []

        for fold in range(1, 11):
            with tf.device('/GPU:0'):
                if not os.path.exists(f"{model_path}/best_loss/fold_{fold}") or not os.path.exists(f"{model_path}/best_loss/fold_{fold}/finished.txt"):
                    print(f"FOLD {fold} DOES NOT EXIST!")
                    continue

                
                output_folder = f"models/128x128/predictions/{'/'.join(parts[3:])}"
                print(output_folder)
                if not os.path.exists(f"{output_folder}/fold_{fold}_raw.csv"):
                    print(model_path)

                    model = model_funcs[model_name](training_config, experiment_name)
                    model.load_weights(f"{model_path}/best_loss/fold_{fold}/{model_name}.weights.h5") # new way
                    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4, beta_1=0.9, beta_2=0.999, epsilon=1e-08),
                        loss=keras.losses.SparseCategoricalCrossentropy(),
                        metrics=[tf.keras.metrics.SparseCategoricalAccuracy()])
                    galaxy_ids = np.empty([0, 1], dtype=str)
                    true_labels = np.empty([0, 1], dtype=float)
                    predictions = np.empty([0, 1], dtype=float)
                    raw = np.empty([0, training_config["NUMBER_OF_CLASSES"]], dtype=float)

                    dataset = get_dataset_with_objids(f"TFDataset/{dataset_name}/all/fold_{fold}/test", 256)

                    for images, labels, objids in tqdm(dataset):
                        results = model.predict(images)
                        galaxy_ids = tf.concat([tf.reshape(objids, [-1, 1]), galaxy_ids], axis=0)
                        true_labels = tf.concat([tf.reshape(labels, [-1, 1]), true_labels], axis=0)
                        predictions = tf.concat([tf.reshape(tf.argmax(results, axis= 1), [-1, 1]), predictions], axis=0)
                        raw = tf.concat([tf.reshape(results, [-1, training_config["NUMBER_OF_CLASSES"]]), raw], axis=0)


                    data_dictionary = {
                        "Id": galaxy_ids.numpy().astype(str).reshape((-1)),
                        "True": true_labels.numpy().reshape((-1)),
                        "Prediction": predictions.numpy().reshape((-1))
                    }

                    for i in range(0, training_config["NUMBER_OF_CLASSES"]):
                        data_dictionary[f"Class_{i}"] = raw.numpy()[:, i].reshape((-1))

                    prediction_dataframe = pd.DataFrame(data_dictionary)

                    Path(output_folder).mkdir(parents=True, exist_ok=True)
                    prediction_dataframe.to_csv(f"{output_folder}/fold_{fold}_raw.csv")

                    del model
                    del dataset
                    del galaxy_ids
                else:
                    print(f"PATH ALREADY EXISTS: {output_folder}/fold_{fold}.csv")
            
            tf.keras.backend.clear_session()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, f1_score, precision_score, recall_score
from scipy.stats import wilcoxon, ttest_rel, ttest_ind
import os
from glob import glob
import pandas as pd
import numpy as np
from IPython.display import HTML, display
from scipy.stats import wilcoxon

def print_metric_pvalue(dataframe, metric_name):
    baseline_acc = dataframe[dataframe["model_name"] == "baseline"].sort_values("fold")[metric_name]
    focal_acc = dataframe[dataframe["model_name"] == "focal"].sort_values("fold")[metric_name]
    stat, pvalue = wilcoxon(baseline_acc, focal_acc)
    stat, paired_pvalue = ttest_rel(baseline_acc, focal_acc)
    print(f"{metric_name.title()} Wilcoxon p-value: {pvalue:.3f}, paired t-test p-value: {paired_pvalue:.3f}")



baseline_models = {
    "cavanagh_no_focal": None,
    "cavanagh_focal_1": "cavanagh_no_focal",
    "cavanagh_focal_5": "cavanagh_no_focal",
    "cavanagh_focal_10": "cavanagh_no_focal",
    "cheng_no_focal": None,
    "cheng_focal_1" : "cheng_no_focal",
    "cheng_focal_5" : "cheng_no_focal",
    "cheng_focal_10" : "cheng_no_focal",
    "best_loss_11class_no_focal": None,
    "best_loss_11class_original": "best_loss_11class_no_focal",
    "best_loss_11class_all_focal": "best_loss_11class_no_focal",
    "top_10_loss_4class_no_focal": None,
    "top_10_loss_4class_original": "top_10_loss_4class_no_focal",
    "top_10_loss_4class_all_focal": "top_10_loss_4class_no_focal",
}

def calc_stat(dataframe, model_name, metric_fn, f1_scores):
    if baseline_models[model_name] == None:
        return "-"

    baseline_data = dataframe[dataframe["model_name"] == baseline_models[model_name]]["f1"]
    try:
        stat, pvalue = metric_fn(baseline_data, f1_scores)

        return f"{pvalue:.3f}, {'pass' if pvalue < 0.05 else 'fail'}"
    except:
        return "Err"


for experiment_name in [
    "paper3_focal_last_Bin_DES",
    "paper3_focal_last_Bin_SDSS",
    "paper3_focal_last_Bin_H_1",
    "paper3_focal_last_Bin_H_2",
    "paper3_focal_last_Bin_H_3"
    ]:
    print(experiment_name)
    model_paths = glob(f"models/128x128/predictions/{experiment_name}/*")


    dataframe_baseline = pd.DataFrame([], columns = ['model_name', 'experiment', 'fold', 'accuracy', 'precision', 'recall', 'f1'])

    def format(x):
        return f"{np.mean(x * 100):.2f} ({np.std(x * 100):.2f})" 


    for path in model_paths:
        parts = path.replace("\\", "/").split("/")
        model_name = parts[4]

        for fold in range(1, 11):
            if(os.path.exists(f"{path}/fold_{fold}_raw.csv".replace("\\", "/"))):
                dataset = pd.read_csv(f"{path}/fold_{fold}_raw.csv".replace("\\", "/"))
                true_labels = dataset["True"].to_numpy().T
                predictions = dataset["Prediction"].to_numpy().T

                new_df = pd.DataFrame([[
                    model_name,
                    experiment_name,
                    fold,
                    accuracy_score(true_labels, predictions),
                    precision_score(true_labels, predictions, average = 'macro'),
                    recall_score(true_labels, predictions, average = 'macro'),
                    f1_score(true_labels, predictions, average = 'macro')
                ]], columns = ['model_name', 'experiment', 'fold', 'accuracy', 'precision', 'recall', 'f1'])

                dataframe_baseline = pd.concat([dataframe_baseline, new_df])

    # print_metric_pvalue(dataframe_baseline, "accuracy")
    # print_metric_pvalue(dataframe_baseline, "f1")

    table = dataframe_baseline.groupby(["model_name", "experiment"]).apply(func= lambda x:
        pd.Series({
            "Folds": len(x["accuracy"]),
            "Accuracy (σ)": format(x["accuracy"]),
            "Precision (σ)": format(x["precision"]),
            "Recall (σ)": format(x["recall"]),
            "F1 (σ)": format(x["f1"]),
            "Wilcoxon": calc_stat(dataframe_baseline, np.array(x["model_name"])[0], wilcoxon, np.array(x["f1"])),
            "Paired": calc_stat(dataframe_baseline, np.array(x["model_name"])[0], ttest_rel, np.array(x["f1"])),
            "Welch": calc_stat(dataframe_baseline, np.array(x["model_name"])[0], ttest_ind, np.array(x["f1"])),
        })
    )

    display(table)


    

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, f1_score, precision_score, recall_score
from scipy.stats import wilcoxon, ttest_rel, ttest_ind
import os
from glob import glob
import pandas as pd
import numpy as np
from IPython.display import HTML, display
from scipy.stats import wilcoxon


for experiment_name in [
    "paper3_focal_ablation_no_focal_baseline_Bin_DES",
    "paper3_focal_ablation_no_focal_baseline_Bin_SDSS",
    "paper3_focal_ablation_no_focal_baseline_Bin_H_1",
    "paper3_focal_ablation_no_focal_baseline_Bin_H_2",
    "paper3_focal_ablation_no_focal_baseline_Bin_H_3"
    ]:
    print(experiment_name)
    model_paths = glob(f"models/128x128/predictions/{experiment_name}/*")


    dataframe_baseline = pd.DataFrame([], columns = ['model_name', 'experiment', 'fold', 'accuracy', 'precision', 'recall', 'f1'])

    def format(x):
        return f"{np.mean(x * 100):.2f} ({np.std(x * 100):.2f})" 


    for path in model_paths:
        parts = path.replace("\\", "/").split("/")
        model_name = parts[4]

        for fold in range(1, 11):
            if(os.path.exists(f"{path}/fold_{fold}_raw.csv".replace("\\", "/"))):
                dataset = pd.read_csv(f"{path}/fold_{fold}_raw.csv".replace("\\", "/"))
                true_labels = dataset["True"].to_numpy().T
                predictions = dataset["Prediction"].to_numpy().T

                new_df = pd.DataFrame([[
                    model_name,
                    experiment_name,
                    fold,
                    accuracy_score(true_labels, predictions),
                    precision_score(true_labels, predictions, average = 'macro'),
                    recall_score(true_labels, predictions, average = 'macro'),
                    f1_score(true_labels, predictions, average = 'macro')
                ]], columns = ['model_name', 'experiment', 'fold', 'accuracy', 'precision', 'recall', 'f1'])

                dataframe_baseline = pd.concat([dataframe_baseline, new_df])

    # print_metric_pvalue(dataframe_baseline, "accuracy")
    # print_metric_pvalue(dataframe_baseline, "f1")

    table = dataframe_baseline.groupby(["model_name", "experiment"]).apply(func= lambda x:
        pd.Series({
            "Folds": len(x["accuracy"]),
            "F1 (σ)": format(x["f1"]),
            "Accuracy (σ)": format(x["accuracy"]),
            "Precision (σ)": format(x["precision"]),
            "Recall (σ)": format(x["recall"]),
        })
    )

    display(table)


    